# `DPR processing` and `Auxip staging` Prefect flow

  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-797
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-798
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-799

## Initialisation

In [ ]:
# Imports
from importlib import reload
import ipywidgets as widgets
import os
import os.path as osp
import prefect
import sys

from resources.widget_utils import deploy_prefect_radio, run_prefect_radio, run_prefect

from rs_common import prefect_utils
from rs_common.prefect_utils import *
import rs_workflows
from rs_workflows.flow_utils import  DprProcessIn, Priority, ProcessingMode, ProcessorEnum, WorkflowType
from rs_workflows.init_pi_db_flow import init_pi_database
from rs_workflows.on_demand_processing import on_demand_cadip_staging

In [ ]:
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard_url = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard_url}")

In [ ]:
deploy_prefect_radio

In [ ]:
run_prefect_radio

In [16]:
dpr_proc = widgets.RadioButtons(
    options=[(proc.name, proc) for proc in ProcessorEnum],
    value=ProcessorEnum.MOCKUP,
    description="DPR processor in this demo:",
    indent=False,
)
dpr_proc

RadioButtons(description='DPR processor in this demo:', options=(('MOCKUP', <ProcessorEnum.MOCKUP: 'mockup'>),…

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_staging(scale=2)

# Init the processor dask cluster. 
# NOTE: use little resources for now because we only call the task tables.
match dpr_proc.value:
    case ProcessorEnum.MOCKUP:
        init_dask_cluster_mockup(scale=1)
    case ProcessorEnum.S1L0 | ProcessorEnum.S3L0:
        init_dask_cluster_l0(scale=1)
    case ProcessorEnum.S1ARD:
        init_dask_cluster_s1ard(scale=1)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
display(dask_cluster_staging)
display(dask_cluster_eopf)

In [ ]:
# Local paths
rs_workflows_parent = Path(rs_workflows.__path__[0]).parent

# Get the prefect share bucket folder
share_bucket, _ = await get_share_bucket()

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    share_bucket.bucket_name,
    share_bucket.bucket_folder,
    "users",
    OWNER_ID,
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")

In [ ]:
# Prefect flow environment arguments
flow_env_args = {
  "env": {
    "owner_id": OWNER_ID,
  },
}

# Create a test collection
CATALOG_COLLECTION_ID = "DPR_PROCESSING_TEST_COLLECTION"
collection = create_test_collection(CATALOG_COLLECTION_ID)

# DPR processing input parameters
dpr_process_in = DprProcessIn(
    env=flow_env_args, 
    processor_name=dpr_proc.value, 
    processor_version="", # NOTE: is it used ?
    dask_cluster_label=cluster_info_eopf.cluster_label,
    pipeline = "todo",
    unit = None,
    priority = Priority.LOW,
    workflow_type = WorkflowType.ON_DEMAND,
    input_products = [],
    generated_product_to_collection_identifier = {"*", CATALOG_COLLECTION_ID},
    auxiliary_product_to_collection_identifier = {"*", CATALOG_COLLECTION_ID},
    processing_mode = [ProcessingMode.ALWAYS],
    start_datetime = None, 
    end_datetime = None,
    satellite=None,
)

## Deploy and run INIT PI DB flow

In [ ]:
%%bash -s "$rs_workflows_parent"
# Deploy the flow
deploy_file=$(realpath "../../sprint27/init_pi_db_flows.yaml")
echo "Deploying '$deploy_file'..."
(cd $1; prefect --no-prompt deploy --prefect-file "$deploy_file" --all)

In [ ]:
# Flow deployment names
pi_deploy = "PI db init/PI db init"
await prefect_utils.wait_for_deployment(pi_deploy)
await run_prefect(pi_deploy, init_pi_database, flow_env_args)

## Deploy Prefect flow

We deploy the Prefect workflows that are implemented in the `rs-client-libraries` git repository.

WARNING: the `rs-client-libraries` source code must be identical in these 3 environments:

  * https://github.com/RS-PYTHON/rs-demo.git (if we deploy using git)
  * This Jupyter environment
  * The Prefect Docker images

In [ ]:
# Flow deployment names
processing_deploy = "dpr-process/DPR processing"
auxip_deploy = "On-demand Auxip staging/Auxip staging"
cadip_deploy = "On-demand Cadip staging/Cadip staging"

In [ ]:
%%bash -s "$rs_workflows_parent" "$deploy_prefect_radio.value"
# Deploy the flows using git
if [[ $2 == "git" ]]; then
    deploy_file=$(realpath "./dpr_processing_flow.yaml")
    echo "Deploying '$deploy_file'..."
    (cd $1; prefect --no-prompt deploy --prefect-file "$deploy_file" --all)
fi

In [ ]:
if deploy_prefect_radio.value == "bucket":

    # Use a subfolder named after the current user
    s3_code_folder = f"users/{OWNER_ID}/code" 

    # Use a specific secret block on the bucket for this subfolder
    code_bucket, _ = await get_share_bucket(s3_code_folder)
    
    # Upload workflows package contents
    await code_bucket.put_directory(local_path = rs_workflows.__path__[0], to_path = "rs_workflows")

    # Reload all rs-client-libraries modules
    for module in list(sys.modules.values()):
        if any(module.__name__.startswith(prefix) for prefix in ["rs_client.", "rs_common.", "rs_workflows."]):
            reload(module)

    # Deploy the flows
    for entrypoint, name in [
        ["on_demand_processing.py:on_demand_processing", "DPR processing"],
        ["on_demand_processing.py:on_demand_auxip_staging", "Auxip staging"],
        ["on_demand_processing.py:on_demand_cadip_staging", "Cadip staging"],
    ]:
        flow = await prefect.flow.from_source(
            source=code_bucket,
            entrypoint=f"rs_workflows/{entrypoint}",
        )
        await flow.deploy(
            name=name,
            work_pool_name=os.environ["PREFECT_WORK_POOL_EOPF"], 
            tags=[""],
            ignore_warnings=True,
        )

In [ ]:
# Wait for deployments
for deploy_name in [processing_deploy, cadip_deploy, auxip_deploy]:
    await prefect_utils.wait_for_deployment(deploy_name)

## Init the L0 demos

In [ ]:
if dpr_proc.value in (ProcessorEnum.S1L0, ProcessorEnum.S3L0):

    if dpr_proc.value == ProcessorEnum.S1L0:
        cadip_collection = "sgs_sentinel1"
        cadip_session = "S1A_20200105072204051312"
    else:
        cadip_collection = "sgs_sentinel3"
        cadip_session = "S3A_20250109134406046340"

    # Stage a cadip session
    params = {
        **flow_env_args,
        "cadip_collection_identifier": cadip_collection,
        "session_identifier": cadip_session,
        "catalog_collection_identifier": CATALOG_COLLECTION_ID,
    }    
    await run_prefect(cadip_deploy, on_demand_cadip_staging, params)

    # Update the input product list of the dpr processing
    dpr_process_in.input_products = {cadip_session: CATALOG_COLLECTION_ID}

Call 'On-demand Cadip staging/Cadip staging' from http://localhost:4200/deployments with:
  - env: {'owner_id': 'jgaucher'}
  - cadip_collection_identifier: sgs_sentinel3
  - session_identifier: S3A_20250109134406046340
  - catalog_collection_identifier: DPR_PROCESSING_TEST_COLLECTION


14:03:57.171 | INFO    | prefect.engine - View at http://prefect-server:4200/runs/flow-run/e9b0fe2c-8818-4a83-9c72-274e8bff1d11

14:03:57.210 | INFO    | Flow run 'perky-kittiwake' - Beginning flow run 'perky-kittiwake' for flow 'On-demand Cadip staging'

14:03:57.212 | INFO    | Flow run 'perky-kittiwake' - View at http://prefect-server:4200/runs/flow-run/e9b0fe2c-8818-4a83-9c72-274e8bff1d11

14:03:57.264 | WARNING | opentelemetry.trace - Overriding of current TracerProvider is not allowed

...

14:05:28.568 | INFO    | Task run 'Cadip staging-de9' - ----- Staging from 'cadip-station' job '334c29a0-9c26-4c1e-a88f-4b195d978632': RUNNING

14:05:30.582 | INFO    | Task run 'Cadip staging-de9' - job_status: {'progress': 10, 'created': '2025-10-10T14:03:57Z', 'started': '2025-10-10T14:03:57Z', 'updated': '2025-10-10T14:05:21Z', 'processID': 'staging', 'type': 'process', 'status': 'running', 'message': 'In progress', 'jobID': '334c29a0-9c26-4c1e-a88f-4b195d978632'}

14:05:30.584 | INFO    | Task run 'Cadip staging-de9' - ----- Staging from 'cadip-station' job '334c29a0-9c26-4c1e-a88f-4b195d978632': RUNNING

14:05:32.594 | INFO    | Task run 'Cadip staging-de9' - job_status: {'progress': 10, 'created': '2025-10-10T14:03:57Z', 'started': '2025-10-10T14:03:57Z', 'updated': '2025-10-10T14:05:21Z', 'processID': 'staging', 'type': 'process', 'status': 'running', 'message': 'In progress', 'jobID': '334c29a0-9c26-4c1e-a88f-4b195d978632'}

14:05:32.595 | INFO    | Task run 'Cadip staging-de9' - ----- Staging from 'cadip-station' job '334c29a0-9c26-4c1e-a88f-4b195d978632': RUNNING

## Common arguments to all processors

In [ ]:
env = 

In [ ]:
main_flow_parameters = {
  "env": {
    "owner_id": OWNER_ID,
  },
  "processor": ProcessorEnum.S3L0,
  "cluster_label": cluster_info_eopf.cluster_label,
  "cadip_collection_identifier": CADIP_COLLECTION_ID,
  "session_identifier": SESSION_ID,
  "catalog_collection_identifier": CATALOG_COLLECTION_ID,
  "s3_payload_template": osp.join(
      s3_config, 
      "s3/s3_l0_demo_payload_dpr_mockup_template.yaml"
  ),
  "s3_output_data": f"{s3_output}/s3",
  "use_dpr_mockup": True,
}

cadip_search_parameters = {
  "env": {
    "owner_id": OWNER_ID,
  },
  "cadip_collection_identifier": CADIP_COLLECTION_ID,
  "session_identifier": SESSION_ID
}

## Run the main Prefect flow

In [ ]:
# Convert to json to trigger prefect flow
params_str = to_json(main_flow_parameters)

<div class="alert alert-block alert-danger">
WARNING: if we deploy using the git repo, by default we use the 'develop' branch from rs-client-libraries, see split_processor_flow.yaml
</div>

In [ ]:
%%bash -s "$main_deploy" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

In [ ]:
# Processed items published to the catalog
ItemCollection(list(catalog_client.get_items(CATALOG_COLLECTION_ID)))

## We can also run only a subflow

In [ ]:
# Convert to json to trigger prefect flow
cadip_params_str = to_json(cadip_search_parameters)

In [ ]:
%%bash -s "$cadip_deploy" "$cadip_params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

<div class="alert alert-block alert-warning">

Note: how to get the job results ?
</div>

## Shutdown the dask clusters

In [ ]:
shutdown = widgets.Checkbox(
    value=False,
    description="Shutdown the dask clusters",
    indent=False
)
shutdown

In [ ]:
if shutdown.value:
    shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)
    shutdown_dask_clusters(dask_gateway_staging, dask_cluster_staging.name)
    close_dask_clusters()
# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

## For testing only: reset the clusters and run the flow locally from Python

In [ ]:
debug_locally = widgets.Checkbox(
    value=False,
    description="Debug locally",
    indent=False
)
debug_locally

In [ ]:
if debug_locally.value:
    shutdown_dask_clusters(dask_gateway_staging, None)
    shutdown_dask_clusters(dask_gateway_eopf, None)
    init_dask_cluster_mockup(scale=2)
    init_dask_cluster_staging(scale=2)

    from resources.dask_utils import *

In [ ]:
if debug_locally.value:

    # Reload all rs-client-libraries modules
    for module in list(sys.modules.values()):
        if any(module.__name__.startswith(prefix) for prefix in ["rs_client.", "rs_common.", "rs_workflows."]):
            reload(module)

    from rs_workflows import on_demand_processing
    results = await on_demand_processing.on_demand_processing(**main_flow_parameters)
    display(results)